# **Neuromorphic computing - LAB 04**
May-June 2026, "Machine learning in applications" course

*Prof. G. Urgese, V. Fra, B. Leto*

---
This notebook is an assignment to be completed and submitted by Thursday 21 at 23:59. The objective is to perform a series of tasks on the miRNA dataset using Spiking Neural Networks (SNNs). Throughout the assignment, you will explore the dataset, apply the required preprocessing and analysis steps, and implement SNN-based methods to address the proposed tasks.
---

In [95]:
!pip install snntorch==0.6.2 --quiet

You should consider upgrading via the 'C:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\ML in Application\LABS\Labs-FP\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [96]:
import torch, torch.nn as nn
import snntorch as snn

### Useful functions

In [97]:
import csv
import numpy as np
import scipy.stats
from sklearn.model_selection import train_test_split

In [98]:
def extract_label(file_name, verbose=False):
    data = {}
    label = []
    with open(file_name, "r") as fin:
        reader = csv.reader(fin, delimiter=',')
        first = True
        for row in reader:
            lbl = row[2]
            if first or "TARGET" in lbl:
                first = False
                continue
            lbl = lbl.replace("TCGA-","")

            label.append(lbl)
            if lbl in data.keys():
                data[lbl] += 1
            else:
                data[lbl] = 1
    if verbose:
        print(f"Number of classes in the dataset = {len(data)}")
        pprint.pprint(data, indent=4)

    return label

In [99]:
def create_dictionary(labels):
    dictionary = {}
    class_names = np.unique(labels)
    for i, name in enumerate(class_names):
        dictionary[name] = i
    return dictionary

In [100]:
def label_processing(labels):
    new_miRna_label = []
    dictionary = create_dictionary(labels)
    for i in labels:
        new_miRna_label.append(dictionary[i])
    return new_miRna_label

### 1.2 Download Dataset

In [101]:
import os
mir_dataset = "https://drive.google.com/drive/folders/1oWWeord8YYvtxIo2Pq2peyx7xOI-1Tmb?usp=sharing"
if "MLinApp_course_data" not in os.listdir("./"):
  ! gdown $mir_dataset -O ./MLinApp_course_data --folder

In [102]:
# Remove the first row and the last column from the feature
miR_label = extract_label("./MLinApp_course_data/tcga_mir_label.csv")
miR_data = np.genfromtxt('./MLinApp_course_data/tcga_mir_rpm.csv', delimiter=',')[1:,0:-1]

In [103]:
number_to_delete = abs(len(miR_label) - miR_data.shape[0])
miR_data = miR_data[number_to_delete:,:]
# Convert labels in number
num_miR_label = label_processing(miR_label)

In [104]:
# Z-score normalization
miR_data = scipy.stats.zscore(miR_data, axis=1)

assert np.isnan(miR_data).sum() == 0

In [105]:
print(miR_data[0], np.min(miR_data))

[ 1.68703834  1.67910068  1.71667838 ... -0.05112508 -0.01854857
  3.38106288] -0.13941802539632334


In [106]:
# log2 normalization <Optional>

miR_data = miR_data + abs(np.min(miR_data)) + 0.001

miR_data = np.log2(miR_data)

In [107]:
# normalization between [0, 255] <Optional>
miR_data = (miR_data - np.min(miR_data)) / (np.max(miR_data) - np.min(miR_data)) * 255

In [108]:
n_classes = np.unique(miR_label).size

print(n_classes)
print(miR_label)

print(num_miR_label)
print(miR_data)

33
['READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'REA

# DataLoading
Define variables for dataloading.

In [109]:
batch_size = 128
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
padded_data = False

Define function to add padding to our data \<Optional\>

In [110]:
import math

def add_pad_data(data):
  miR_data = data
  c_int = math.ceil(np.sqrt(len(miR_data[0])))
  pad = c_int ** 2 - len(miR_data[0])
  pad_width = (0, pad)

  padded_miR_data = np.zeros((miR_data.shape[0], miR_data.shape[1] + pad_width[1]))

  for i in range(len(miR_data)):
    padded_miR_data[i] = np.pad(miR_data[i], pad_width, mode='constant')

  # reshape shape[1] into (c_int, c_int)

  dim = int(np.sqrt(len(padded_miR_data[0])))
  padded_miR_data = padded_miR_data.reshape((padded_miR_data.shape[0],1, dim, dim))

  return padded_miR_data

## TODO: Generate subset based on top N most frequent labels \<Optional\>

From the dataset extract the 10 most frequent classes

In [111]:
N = 10

In [112]:
# TODO: Write here your the code for identifying the most frequent classes


# Assuming 'labels' is a list of class labels
from collections import Counter
label_counts = Counter(num_miR_label)
most_common_labels = label_counts.most_common(N)
most_frequent_classes = [label for label, count in most_common_labels]
print(f"The {N} most frequent classes are: {most_frequent_classes}")

The 10 most frequent classes are: [2, 11, 30, 28, 9, 16, 22, 14, 17, 19]


## TODO: Dimensionality analysis and reduction using Principal Component Analysis  \<Optional\>

---

(PCA) on train_data.

Keep only features that preserve 99% of the variance.

For further information, please look at the documentation available at https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

In [113]:
from sklearn.decomposition import PCA

In [114]:

original_features = miR_data.shape[1]

pca = PCA(n_components=0.99)
miR_data = pca.fit_transform(miR_data)

print(f"Features: {original_features} → {miR_data.shape[1]} (PCA, 99% variance)")
print(f"Explained variance retained: {pca.explained_variance_ratio_.sum():.4f}")


Features: 1881 → 85 (PCA, 99% variance)
Explained variance retained: 0.9902


## Create DataLoader

In [115]:
# usefull if you want to represent your data as images <Optional>

miR_data = add_pad_data(miR_data)
padded_data = True

In [116]:
train_data, val_data, train_label, val_label = train_test_split(miR_data, num_miR_label, test_size=0.20, random_state=42)

In [117]:
from torch.utils.data import TensorDataset, DataLoader
miR_train = torch.Tensor(train_data)
miR_train_label = torch.Tensor(train_label)
miR_dataset_train = TensorDataset(miR_train, miR_train_label)

miR_val = torch.Tensor(val_data)
miR_val_label = torch.Tensor(val_label)
miR_dataset_val = TensorDataset(miR_val, miR_val_label)

train_loader = DataLoader(miR_dataset_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(miR_dataset_val, batch_size=batch_size)

# Define Network
Let's compare the performance of a pair of networks both with and without population coding.


Each group should try the assigned values.


In [118]:
from snntorch import surrogate

# network parameters
if padded_data:
  num_inputs = train_data.shape[2] ** 2
else:
  num_inputs = train_data.shape[1]

num_hidden = 128       # Group B
num_outputs = n_classes

# temporal dynamics
num_steps = 10         # Group B

# spiking neuron parameters
beta = 0.9             # Group D
grad = surrogate.fast_sigmoid()


## Without population coding

In [119]:
first_layer_neuron = snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True)
second_layer_neuron =  snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)

In [120]:
# Lapicque neurons (RC circuit model, beta equivalent to Leaky)
first_layer_neuron = snn.Lapicque(beta=beta, spike_grad=grad, init_hidden=True)
second_layer_neuron = snn.Lapicque(beta=beta, spike_grad=grad, init_hidden=True, output=True)


In [121]:
# RLeaky neurons (recurrent connections with scalar weight V)
first_layer_neuron = snn.RLeaky(beta=beta, V=0.5, all_to_all=False, spike_grad=grad, init_hidden=True)
second_layer_neuron = snn.RLeaky(beta=beta, V=0.5, all_to_all=False, spike_grad=grad, init_hidden=True, output=True)


In [122]:
# standard network
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(num_inputs, num_hidden),
                    first_layer_neuron,
                    nn.Linear(num_hidden, num_outputs),
                    second_layer_neuron
                    ).to(device)

## Next Step: define your own network

In [123]:
# Custom network: 3-layer, wider (512→256), NO BatchNorm (disrupts spike thresholds)
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(num_inputs, 512),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(512, 256),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(256, num_outputs),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)
).to(device)


## With population coding


In [124]:
neurons_per_classes = 50   # Group B
pop_outputs = n_classes * neurons_per_classes


In [125]:
first_layer_neuron =  snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True)
second_layer_neuron =  snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)

In [126]:
# Lapicque neurons (RC circuit model, beta equivalent to Leaky)
first_layer_neuron = snn.Lapicque(beta=beta, spike_grad=grad, init_hidden=True)
second_layer_neuron = snn.Lapicque(beta=beta, spike_grad=grad, init_hidden=True, output=True)


In [127]:
# RLeaky neurons (recurrent connections with scalar weight V)
first_layer_neuron = snn.RLeaky(beta=beta, V=0.5, all_to_all=False, spike_grad=grad, init_hidden=True)
second_layer_neuron = snn.RLeaky(beta=beta, V=0.5, all_to_all=False, spike_grad=grad, init_hidden=True, output=True)


In [128]:
# standard network with population coding

net_pop = nn.Sequential(nn.Flatten(),
                        nn.Linear(num_inputs, num_hidden),
                        first_layer_neuron,
                        nn.Linear(num_hidden, pop_outputs),
                        second_layer_neuron
                        ).to(device)

## Next Step: Define your own network with population coding

In [129]:
# Custom network with population coding: 3-layer, wider (512→256), NO BatchNorm
net_pop = nn.Sequential(
    nn.Flatten(),
    nn.Linear(num_inputs, 512),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(512, 256),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(256, pop_outputs),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)
).to(device)


# Training
## Without population coding
Define the optimizer and loss function. Here, we use the MSE Count Loss, which counts up the total number of output spikes at the end of the simulation run.

The correct class has a target firing probability of 100%, and incorrect classes are set to 0%.

In [130]:
import snntorch.functional as SF

learning_rate = 2e-3   # Group C

optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, betas=(0.9, 0.999))
loss_fn = SF.mse_count_loss(correct_rate=1.0, incorrect_rate=0.0)


We will also define a simple test accuracy function that predicts the correct class based on the neuron with the highest spike count.

In [131]:
from snntorch import utils

def test_accuracy(data_loader, net, num_steps, population_code=False, num_classes=False):
  with torch.no_grad():
    total = 0
    acc = 0
    net.eval()

    data_loader = iter(data_loader)
    for data, targets in data_loader:
      data = data.to(device)
      targets = targets.to(device)
      utils.reset(net)
      spk_rec, _ = net(data)

      if population_code:
        acc += SF.accuracy_rate(spk_rec.unsqueeze(0), targets, population_code=True, num_classes=n_classes) * spk_rec.size(1)
      else:
        acc += SF.accuracy_rate(spk_rec.unsqueeze(0), targets) * spk_rec.size(1)

      total += spk_rec.size(1)

  return acc/total

Let's run the training loop.

In [132]:
from snntorch import backprop

num_epochs = 20

for inst in snn.RLeaky.instances:
    if isinstance(inst, snn.RLeaky) and not hasattr(inst, 'spk'):
        inst.spk = torch.zeros(1, device=device)
        inst.mem = torch.zeros(1, device=device)

# training loop
for epoch in range(num_epochs):

    avg_loss = backprop.BPTT(net, train_loader, num_steps=num_steps,
                          optimizer=optimizer, criterion=loss_fn, time_var=False, device=device)

    print(f"Epoch: {epoch}")
    print(f"Test set accuracy: {test_accuracy(test_loader, net, num_steps)*100:.3f}%\n")


Epoch: 0
Test set accuracy: 5.181%

Epoch: 1
Test set accuracy: 6.309%

Epoch: 2
Test set accuracy: 11.929%

Epoch: 3
Test set accuracy: 11.523%

Epoch: 4
Test set accuracy: 10.231%

Epoch: 5
Test set accuracy: 12.294%

Epoch: 6
Test set accuracy: 14.224%

Epoch: 7
Test set accuracy: 12.271%

Epoch: 8
Test set accuracy: 16.817%

Epoch: 9
Test set accuracy: 15.591%

Epoch: 10
Test set accuracy: 17.110%

Epoch: 11
Test set accuracy: 15.537%

Epoch: 12
Test set accuracy: 17.761%

Epoch: 13
Test set accuracy: 13.812%

Epoch: 14
Test set accuracy: 15.103%

Epoch: 15
Test set accuracy: 16.763%

Epoch: 16
Test set accuracy: 23.008%

Epoch: 17
Test set accuracy: 18.852%

Epoch: 18
Test set accuracy: 19.421%

Epoch: 19
Test set accuracy: 21.900%



## With population coding

In [133]:
learning_rate = 2e-3   # Group C

loss_fn = SF.mse_count_loss(correct_rate=1.0, incorrect_rate=0.0, population_code=True, num_classes=n_classes)
optimizer = torch.optim.Adam(net_pop.parameters(), lr=learning_rate, betas=(0.9, 0.999))


In [134]:
num_epochs = 20

for inst in snn.RLeaky.instances:
    if isinstance(inst, snn.RLeaky) and not hasattr(inst, 'spk'):
        inst.spk = torch.zeros(1, device=device)
        inst.mem = torch.zeros(1, device=device)

# training loop
for epoch in range(num_epochs):

    avg_loss = backprop.BPTT(net_pop, train_loader, num_steps=num_steps,
                            optimizer=optimizer, criterion=loss_fn, time_var=False, device=device)

    print(f"Epoch: {epoch}")
    print(f"Test set accuracy: {test_accuracy(test_loader, net_pop, num_steps, population_code=True, num_classes=n_classes)*100:.3f}%\n")


Epoch: 0
Test set accuracy: 49.319%

Epoch: 1
Test set accuracy: 56.800%

Epoch: 2
Test set accuracy: 58.146%

Epoch: 3
Test set accuracy: 57.940%

Epoch: 4
Test set accuracy: 58.281%

Epoch: 5
Test set accuracy: 59.073%

Epoch: 6
Test set accuracy: 59.622%

Epoch: 7
Test set accuracy: 60.224%

Epoch: 8
Test set accuracy: 59.313%

Epoch: 9
Test set accuracy: 59.747%

Epoch: 10
Test set accuracy: 60.018%

Epoch: 11
Test set accuracy: 59.377%

Epoch: 12
Test set accuracy: 60.066%

Epoch: 13
Test set accuracy: 59.052%

Epoch: 14
Test set accuracy: 60.538%

Epoch: 15
Test set accuracy: 60.446%

Epoch: 16
Test set accuracy: 58.710%

Epoch: 17
Test set accuracy: 60.332%

Epoch: 18
Test set accuracy: 60.592%

Epoch: 19
Test set accuracy: 59.806%




### Results (33 classes)

|  | Standard network | Custom network (3-layer, 512→256) |
|---|---|---|
| **Without population coding** | ~15% | **~23%** (peak epoch 16) |
| **With population coding** | ~30% | **~61%** (peak epoch 18) |




# Conclusion
The performance boost from population coding may start to fade as the number of time steps increases. But it may also be preferable to increasing time steps as PyTorch is optimized for handling matrix-vector products, rather than sequential, step-by-step operations over time.

* For a detailed tutorial of spiking neurons, neural nets, encoding, and training using neuromorphic datasets, check out the
[snnTorch tutorial series](https://snntorch.readthedocs.io/en/latest/tutorials/index.html).
* For more information on the features of snnTorch, check out the [documentation at this link](https://snntorch.readthedocs.io/en/latest/).
* If you have ideas, suggestions or would like to find ways to get involved, then [check out the snnTorch GitHub project here.](https://github.com/jeshraghian/snntorch)